In [ ]:
pip install lightgbm

In [ ]:
pip install mlflow

In [ ]:
pip install xgboost

In [ ]:
pip install tensorflow

In [ ]:
%fs ls dbfs/FileStore/processed_data/


In [ ]:
input_path_csv = '/dbfs/FileStore/processed_data/DBT_df_encoded.csv'

# Load the CSV file into a Spark DataFrame
DBT_spark_df_loaded_csv = spark.read.csv(input_path_csv, header=True, inferSchema=True)

# Convert the Spark DataFrame to a Pandas DataFrame
DBT_df_encoded = DBT_spark_df_loaded_csv.toPandas()

# Display the Pandas DataFrame
DBT_df_encoded.head()

### Predicting Medication Adherence

The primary goal of the model is to predict whether a patient is likely to adhere to their medication regimen. 

This is based on the mpr80 column, where:

? mpr80 = 1: Indicates that the patient's Medication Possession Ratio (MPR) is 80% or higher, meaning they are adherent to their medication.

? mpr80 = 0: Indicates that the patient's MPR is below 80%, meaning they are not adherent to their medication.

Identifying patients who are at high risk of non-adherence (i.e., those predicted to have mpr80 = 0).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV

# Adding new features
DBT_df_encoded['total_gaps'] = DBT_df_encoded['gaps_bl'] + DBT_df_encoded['leading_gap']
DBT_df_encoded['high_copay'] = (DBT_df_encoded['copay_tc_nm'] > DBT_df_encoded['copay_tc_nm'].mean()).astype(int)

# Step 4: Define target and features
X = DBT_df_encoded.drop(columns=['mpr', 'mpr80', 'PAT_ID'])  # Drop unnecessary columns including mpr and mpr80
y = DBT_df_encoded['mpr80']  # Use mpr80 as the target variable

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier


In [ ]:
with mlflow.start_run(run_name="XGBoost Classifier DBT"):
    # Initialize XGBoost Classifier
    xgb_model = XGBClassifier(random_state=42)
    
    # Train the model
    xgb_model.fit(X_train, y_train)
    
    # Predicting on test set
    y_pred = xgb_model.predict(X_test)
    y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    
    # Log the model
    mlflow.sklearn.log_model(xgb_model, "model")

    print(f"XGBoost - Accuracy: {accuracy:.2f}, Precision: {precision:.2f}, Recall: {recall:.2f}, F1 Score: {f1:.2f}, ROC AUC Score: {roc_auc:.2f}")


In [ ]:
with mlflow.start_run(run_name="LightGBM Classifier DBT"):
    # Initialize LGBM Classifier
    lgbm_model = LGBMClassifier(random_state=42)
    
    # Train the model
    lgbm_model.fit(X_train, y_train)
    
    # Predicting on test set
    y_pred = lgbm_model.predict(X_test)
    y_pred_proba = lgbm_model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    
    # Log the model
    mlflow.sklearn.log_model(lgbm_model, "model")

    print(f"LGBM - Accuracy: {accuracy:.2f}, Precision: {precision:.2f}, Recall: {recall:.2f}, F1 Score: {f1:.2f}, ROC AUC Score: {roc_auc:.2f}")


In [ ]:
with mlflow.start_run(run_name="Logistic Regression DBT"):
    # Initialize Logistic Regression
    logreg_model = LogisticRegression(random_state=42, max_iter=1000)
    
    # Train the model
    logreg_model.fit(X_train, y_train)
    
    # Predicting on test set
    y_pred = logreg_model.predict(X_test)
    y_pred_proba = logreg_model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    
    # Log the model
    mlflow.sklearn.log_model(logreg_model, "model")

    print(f"Logistic Regression - Accuracy: {accuracy:.2f}, Precision: {precision:.2f}, Recall: {recall:.2f}, F1 Score: {f1:.2f}, ROC AUC Score: {roc_auc:.2f}")


In [ ]:
with mlflow.start_run(run_name="Random Forest Classifier DBT"):
    # Initialize RandomForestClassifier
    rf_model = RandomForestClassifier(random_state=42)
    
    # Train the model
    rf_model.fit(X_train, y_train)
    
    # Predicting on test set
    y_pred = rf_model.predict(X_test)
    y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    
    # Log the model
    mlflow.sklearn.log_model(rf_model, "model")

    print(f"Random Forest - Accuracy: {accuracy:.2f}, Precision: {precision:.2f}, Recall: {recall:.2f}, F1 Score: {f1:.2f}, ROC AUC Score: {roc_auc:.2f}")


In [ ]:
import mlflow.tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import AUC

# Start an MLflow run
with mlflow.start_run(run_name="Neural Network Classifier DBT"):
    # Define the neural network architecture
    model = Sequential([
        Dense(64, input_shape=(X_train.shape[1],), activation='relu'),
        Dropout(0.5),
        Dense(32, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])

    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss=BinaryCrossentropy(),
                  metrics=['accuracy', AUC(name='auc')])

    # Train the model
    history = model.fit(X_train, y_train, 
                        epochs=10, 
                        batch_size=32, 
                        validation_split=0.2,
                        verbose=2)

    # Evaluate the model on the test set
    test_loss, test_accuracy, test_auc = model.evaluate(X_test, y_test, verbose=0)
    
    # Make predictions
    y_pred_proba = model.predict(X_test)
    y_pred = (y_pred_proba > 0.5).astype("int32")

    # Calculate metrics
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Log metrics
    mlflow.log_metric("accuracy", test_accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", test_auc)

    # Log the model
    mlflow.tensorflow.log_model(model, "model")

    print(f"Neural Network - Accuracy: {test_accuracy:.2f}, Precision: {precision:.2f}, Recall: {recall:.2f}, F1 Score: {f1:.2f}, ROC AUC Score: {test_auc:.2f}")
